# CardioVis — Features_timestamps trim + continuous frames

Notebook này:

1. Đọc `Cardiovis-related/Features_timestamps.xlsx`
2. Hiểu thời gian Excel `HH:MM:SS` là **MM:SS** bị Excel đọc sai (`hour*60 + minute` giây)
3. Lookup video trong `My Drive/3) Videos` theo tên file
4. Trim clip ~10s vào `stage_outputs/clips/{dat_kim_goc,rach_nhi}/`
5. Extract frames (`fps=20`) vào `stage_outputs/frames/`, **đánh số liên tục** sau batch cũ
6. Sample `1/20` sang `stage_outputs/frame_samples_1_per_20/`

**Không** tạo lại `*_full.mp4`.

### Cách chạy
1. Upload `Features_timestamps.xlsx` và `features_timestamps_trim_pipeline.py` vào `My Drive/Cardiovis-related/` (nếu chưa có).
2. Set `DRY_RUN = True` lần đầu để kiểm tra resolve path.
3. Đổi `DRY_RUN = False` và chạy lại để encode.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount("/content/drive")

CARDIOVIS_RELATED = Path("/content/drive/MyDrive/Cardiovis-related")
SCRIPT_PATH = CARDIOVIS_RELATED / "features_timestamps_trim_pipeline.py"
XLSX_PATH = CARDIOVIS_RELATED / "Features_timestamps.xlsx"

for required in (SCRIPT_PATH, XLSX_PATH):
    if not required.exists():
        raise FileNotFoundError(
            f"Missing {required}. Upload it to Cardiovis-related on Drive first."
        )

# Ensure ffmpeg + pandas/openpyxl
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

subprocess.run(["pip", "install", "-q", "pandas", "openpyxl"], check=True)
print("Ready.")
print("Script:", SCRIPT_PATH)
print("Excel :", XLSX_PATH)


In [ ]:
import os

# ---- FLAGS ----
DRY_RUN = True          # True = chỉ check path / report, không encode
OVERWRITE = False
EXTRACT_FPS = 20
SAMPLE_EVERY_N_FRAMES = 20

os.environ["CARDIOVIS_RELATED_DIR"] = "/content/drive/MyDrive/Cardiovis-related"
os.environ["ROOT_VIDEOS_DIR"] = "/content/drive/MyDrive/3) Videos"
os.environ["FEATURES_TIMESTAMPS_XLSX"] = str(XLSX_PATH)
os.environ["DRY_RUN"] = "1" if DRY_RUN else "0"
os.environ["OVERWRITE"] = "1" if OVERWRITE else "0"
os.environ["EXTRACT_FPS"] = str(EXTRACT_FPS)
os.environ["SAMPLE_EVERY_N_FRAMES"] = str(SAMPLE_EVERY_N_FRAMES)

print("DRY_RUN =", DRY_RUN)
!python "$SCRIPT_PATH"


### Sau dry-run

Kiểm tra `stage_outputs/reports/features_timestamps_task_results.csv` trên Drive.
Nếu resolve path OK, set `DRY_RUN = False` ở cell trên và chạy lại.

Outputs mong đợi:
- `stage_outputs/clips/dat_kim_goc/*.mp4`, `clips/rach_nhi/*.mp4`
- `stage_outputs/frames/{stage}/{stage}_full_frame_XXXXXXXX.jpg` (số liên tục)
- `stage_outputs/frame_samples_1_per_20/{stage}/` (mỗi 20 frames)
- `stage_outputs/reports/features_frame_index_map.csv`
